In [27]:
import mlflow
import os
from dotenv import load_dotenv

load_dotenv()

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

EXPERIMENT_NAME = 'FINAL_PROJECT'

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://storage.yandexcloud.net"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

if mlflow.get_experiment_by_name(name=EXPERIMENT_NAME):
    experiment_id = dict(mlflow.get_experiment_by_name(name=EXPERIMENT_NAME))['experiment_id']
    mlflow.set_experiment(experiment_id=experiment_id)
else:
    mlflow.set_experiment(EXPERIMENT_NAME)
    experiment_id = dict(mlflow.get_experiment_by_name(name=EXPERIMENT_NAME))['experiment_id']
    mlflow.set_experiment(experiment_id=experiment_id)

# ALS матрица

In [39]:
import pandas as pd

events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')

events = pd.concat([events_train, events_test])

In [40]:
#кодирование для матрицы

from sklearn.preprocessing import LabelEncoder
import joblib

user_encoder = LabelEncoder()
user_encoder.fit(events['user_id'])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

item_encoder = LabelEncoder()
item_encoder.fit(events['item_id'])
events_train['item_id_enc'] = item_encoder.transform(events_train['item_id'])
events_test['item_id_enc'] = item_encoder.transform(events_test['item_id'])

In [ ]:

joblib.dump(user_encoder, 'user_encoder.pkl')
joblib.dump(item_encoder, 'item_encoder.pkl')
with mlflow.start_run(run_name='encoders_save', experiment_id=experiment_id) as run:
    mlflow.log_artifact('item_encoder.pkl')
    mlflow.log_artifact('user_encoder.pkl')

In [41]:
import scipy
import numpy as np

events_train_view_addtocart = events_train[events_train['event'].isin(['view', 'addtocart'])].copy()

weight_map = {'view': 0.3, 'addtocart': 1.0}
events_train_view_addtocart['weight'] = events_train_view_addtocart['event'].map(weight_map)
interactions_train = (events_train_view_addtocart.groupby(['user_id_enc', 'item_id_enc'])['weight'].max().reset_index())

n_users = len(user_encoder.classes_)
n_items = len(item_encoder.classes_)

user_item_matrix_train = scipy.sparse.csr_matrix(
    (interactions_train['weight'],
     (interactions_train['user_id_enc'], interactions_train['item_id_enc'])),
    shape=(n_users, n_items),
    dtype=np.float32
)

In [5]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=42)
als_model.fit(user_item_matrix_train)

joblib.dump(als_model, 'als_model.pkl')

100%|██████████| 50/50 [02:09<00:00,  2.60s/it]


['als_model.pkl']

In [4]:
als_model = joblib.load('als_model.pkl')

/home/mle-user/mle_projects/mle-final/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations = als_model.recommend(
    user_ids_encoded, 
    user_item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, N=100)

# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])

als_recommendations.to_parquet('als_recommendations.parquet')

# Контентная матрица (не используется)

In [10]:
import pandas as pd

item_categories = pd.read_parquet('item_categories.parquet')

In [16]:
#кодирование itemid для матрицы

from sklearn.preprocessing import LabelEncoder
import joblib

cat_item_enc = LabelEncoder()
item_categories['item_id_enc'] = cat_item_enc.fit_transform(item_categories['item_id'])
joblib.dump(cat_item_enc, 'cat_item_encoder.pkl')

['cat_item_encoder.pkl']

In [ ]:
#кодивование category для матрицы

cat_enc = LabelEncoder()
item_categories['category_enc'] = cat_enc.fit_transform(item_categories['category'])
joblib.dump(cat_enc, 'cat_encoder.pkl')

['cat_encoder.pkl']

In [18]:
import scipy
import numpy as np

item_enc = LabelEncoder()
item_categories['item_id_enc'] = item_enc.fit_transform(item_categories['item_id'])
cat_enc = LabelEncoder()
item_categories['category_enc'] = cat_enc.fit_transform(item_categories['category'])

n_items = item_categories['item_id_enc'].nunique()
n_categories = item_categories['category_enc'].nunique()

category_csr = scipy.sparse.csr_matrix(
    (np.ones(len(item_categories)),
    (item_categories['item_id_enc'], item_categories['category_enc'])),
    shape=(n_items, n_categories),
    dtype=np.int8)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances

user_ids_encoded = range(len(user_encoder.classes_))
results = []

for user in user_ids_encoded: 
    user_id = user
    user_events = interactions_train.query("user_id_enc == @user_id")[["item_id_enc", "weight"]]
    if len(user_events) == 0:
        continue 
    user_items_categories_csr = category_csr[user_events["item_id_enc"].values]

    user_weights = user_events["weight"].to_numpy()
    user_weights = np.expand_dims(user_weights, axis=1)

    user_items_categories_weighted = user_items_categories_csr.multiply(user_weights)

    user_category_scores = np.asarray(user_items_categories_weighted.mean(axis=0))

    # вычисляем сходство между вектором пользователя и векторами по книгам
    similarity_scores = euclidean_distances(category_csr, user_category_scores)

    # преобразуем в одномерный массив
    similarity_scores = similarity_scores.flatten()

    if len(user_events) <= 2:
        k = 5
    else:
        k = 100

    top_k_indices = np.argsort(similarity_scores)[:k]
    top_k_scores = similarity_scores[top_k_indices]

    for item_id, score in zip(top_k_indices, top_k_scores):
        results.append({'user_id': user_id, 'item_id': item_id, 'cnt_score': score})
        
content_recommendations = pd.DataFrame(results)
content_recommendations.to_parquet('content_recommendations.parquet')


# Построение дополнительных признаков

признаки:
1. общее кол-во интеракций у юзера
2. общее кол-во интеракций у айтема
3. общее кол-во просмотров у юзера
4. общее кол-во покупок у юзера
5. общее кол-во просмотров у айтема
6. общее кол-во покупок у айтема
7. категория айтема

In [6]:
import pandas as pd

items = pd.read_parquet('item_categories.parquet')
events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')

In [14]:
# просмотры
user_total_views_train = events_train[events_train['event'] == 'view'].groupby('user_id').size().reset_index(name='user_views_count')
user_total_views_test = events_test[events_test['event'] == 'view'].groupby('user_id').size().reset_index(name='user_views_count')

item_total_views_train = events_train[events_train['event'] == 'view'].groupby('item_id').size().reset_index(name='item_views_count')
item_total_views_test = events_test[events_test['event'] == 'view'].groupby('item_id').size().reset_index(name='item_views_count')

# добавления в корзину
user_total_addtocart_train = events_train[events_train['event'] == 'addtocart'].groupby('user_id').size().reset_index(name='user_addtocart_count')
user_total_addtocart_test = events_test[events_test['event'] == 'addtocart'].groupby('user_id').size().reset_index(name='user_addtocart_count')

item_total_addtocart_train = events_train[events_train['event'] == 'addtocart'].groupby('item_id').size().reset_index(name='item_addtocart_count')
item_total_addtocart_test = events_test[events_test['event'] == 'addtocart'].groupby('item_id').size().reset_index(name='item_addtocart_count')

# все события
user_total_events_train = events_train.groupby('user_id').size().reset_index(name='user_events_count')
user_total_events_test = events_test.groupby('user_id').size().reset_index(name='user_events_count')

item_total_events_train = events_train.groupby('item_id').size().reset_index(name='item_events_count')
item_total_events_test = events_test.groupby('item_id').size().reset_index(name='item_events_count')

# Двухстадийный подход

In [15]:
import pandas as pd

events_train = pd.read_parquet('events_train.parquet')
events_test = pd.read_parquet('events_test.parquet')
als_recommendations = pd.read_parquet('als_recommendations.parquet')

In [16]:
# задаём точку разбиения
split_date_for_labels = pd.to_datetime("2015-08-15")

split_date_for_labels_idx = events_test["date"] < split_date_for_labels
events_labels = events_test[split_date_for_labels_idx].copy()
events_test_2 = events_test[~split_date_for_labels_idx].copy()

In [17]:
# добавляем таргет к кандидатам со значением:
# — 1 для тех item_id, которые пользователь прочитал
# — 0, для всех остальных 

events_train_addtocart = events_train[events_train['event'] == 'addtocart'][['user_id', 'item_id']].copy()
events_train_addtocart['target'] = 1
candidates = als_recommendations.merge(events_train_addtocart[["user_id", "item_id", "target"]], 
                              on=['user_id', 'item_id'],
                              how='left')
candidates["target"] = candidates["target"].fillna(0).astype("int")

# в кандидатах оставляем только тех пользователей, у которых есть хотя бы один положительный таргет
candidates_to_sample = candidates.groupby("user_id").filter(lambda x: x["target"].sum() > 0)

# для каждого пользователя оставляем только 4 негативных примера
negatives_per_user = 4
candidates_for_train = pd.concat([
    candidates_to_sample[candidates_to_sample['target'] == 1],
    candidates_to_sample.query("target == 0") \
        .groupby("user_id") \
        .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=42))
    ])

candidates_to_rank = als_recommendations[als_recommendations["user_id"].isin(events_test_2["user_id"].drop_duplicates())]

/tmp/ipykernel_2636/2430161498.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=42))


In [18]:
# совмещение всех признаков для обучения

items = pd.read_parquet('item_categories.parquet')

candidates_for_train = (
    candidates_for_train
    .merge(user_total_views_train, on='user_id', how='left')
    .merge(user_total_addtocart_train, on='user_id', how='left')
    .merge(item_total_views_train, on='item_id', how='left')
    .merge(item_total_addtocart_train, on='item_id', how='left')
    .merge(user_total_events_train, on='user_id', how='left')
    .merge(item_total_events_train, on='item_id', how='left')
    .merge(items, on='item_id', how='left')
)

candidates_to_rank = (
    candidates_to_rank
    .merge(user_total_views_test, on='user_id', how='left')
    .merge(user_total_addtocart_test, on='user_id', how='left')
    .merge(item_total_views_test, on='item_id', how='left')
    .merge(item_total_addtocart_test, on='item_id', how='left')
    .merge(user_total_events_test, on='user_id', how='left')
    .merge(item_total_events_test, on='item_id', how='left')
    .merge(items, on='item_id', how='left')
)

In [37]:
# предобработка данных перед обучением

cols_to_fix = ['user_views_count', 'item_views_count', 'user_addtocart_count', 
               'item_addtocart_count', 'user_events_count', 'item_events_count']

candidates_for_train[cols_to_fix] = candidates_for_train[cols_to_fix].fillna(0).astype('int32')
candidates_for_train['target'] = candidates_for_train['target'].astype('int8')

candidates_to_rank[cols_to_fix] = candidates_to_rank[cols_to_fix].astype('int32')

candidates_for_train = candidates_for_train.fillna(0)
candidates_to_rank = candidates_to_rank.fillna(0)

candidates_for_train = candidates_for_train.rename(columns={'score': 'als_score'})
candidates_ro_rank = candidates_to_rank.rename(columns={'score': 'als_score'})

In [ ]:
from catboost import CatBoostClassifier, Pool

# задаём имена колонок признаков и таргета
features = ['als_score','user_views_count', 'item_views_count', 'user_addtocart_count', 'item_addtocart_count', 'user_events_count', 'item_events_count', 'category']
cat_features = ['category']
target = 'target'

# создаём Pool
train_data = Pool(
    data=candidates_for_train[features],
    cat_features=cat_features,
    label=candidates_for_train[target])

# инициализируем модель CatBoostClassifier
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0,
)

# тренируем модель
cb_model.fit(train_data)

KeyError: "['score'] not in index"

In [32]:
inference_data = Pool(
    data=candidates_to_rank[features],
    cat_features=cat_features
    )
predictions = cb_model.predict_proba(inference_data)

In [33]:
candidates_to_rank["cb_score"] = predictions[:, 1]

# для каждого пользователя проставим rank, начиная с 1 — это максимальный cb_score
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])
candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1

max_recommendations_per_user = 100
final_recommendations = candidates_to_rank.query("rank <= @max_recommendations_per_user")

In [43]:
final_recommendations = final_recommendations.rename(columns={'score': 'als_score'})

In [34]:
# персональные ALS

def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")
    
    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)
    
    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()
    
    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common

In [35]:
def compute_cls_metrics(events_recs_for_binary_metrics):
    
    groupper = events_recs_for_binary_metrics.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fp"].sum())
    precision = precision.fillna(0).mean()
    
    # recall = tp / (tp + fn)
    recall = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fn"].sum())
    recall = recall.fillna(0).mean()

    return precision, recall

In [44]:
# итоговые

# для экономии ресурсов оставим события только тех пользователей, 
# для которых следует оценить рекомендации
events_inference = pd.concat([events_train, events_labels])
events_inference = events_inference[events_inference["user_id"].isin(events_test_2["user_id"].drop_duplicates())]

cb_events_recs_for_binary_metrics_5 = process_events_recs_for_binary_metrics(
    events_inference,
    events_test_2,
    final_recommendations.rename(columns={"cb_score": "score"}), 
    top_k=5)

cb_precision_5, cb_recall_5 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

print(f"precision: {cb_precision_5}, recall: {cb_recall_5}")

#precision: 0.010776814550399458, recall: 0.01374030080313546

Common users: 171295
precision: 0.0002701584460692844, recall: 0.0005040838502089008


/tmp/ipykernel_2636/3746541708.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


In [45]:
cb_model.get_feature_importance(prettified=True)

,Feature Id,Importances
0,item_addtocart_count,34.959437
1,score,25.948192
2,category,12.155922
3,item_events_count,9.479252
4,item_views_count,8.507963
5,user_addtocart_count,3.343818
6,user_views_count,2.926942
7,user_events_count,2.678474
